# Predicting heart disease usign machine learning

This notebook looks into using various Python-based machine learning and dat science libraries in an attempt to build a machine learning model capable of predicting whether or not someone has heart disease based on their medical attribute.

We're going to take the following approach:
1. Problem definition
2. Data
3. Evaludation
4. Features
5. Modeling
6. Experimentation

## 1. Problem Definition
In a statement,
> Given clinical parameteres about a patient, can we predict whether or not they have heart disease?

## 2. Data
The original data came from the Cleavland data from the UCI Machine Learning Repository. https://archive.ics.uci.edu/ml/datasets/heart+Disease

There is also a version of it available on Kaggle. https://www.kaggle.com/datasets/sumaiyatasmeem/heart-disease-classification-dataset



## 3. Evaluation
> If we can reach 95% accuracy at predicting whether or not a patient has heart disease during the proof of concept, we'll pursue the project

## 4. Features
This is where you'll get different information about each of the features in your data. You can do this via doing your own research (such as looking at the links above) or by talking to a subject matter expert (someone who knows about the dataset).

Create data dictionary

1. age - age in years
2. sex - (1 = male; 0 = female)
3. cp - chest pain type
* 0: Typical angina: chest pain related decrease blood supply to the heart
* 1: Atypical angina: chest pain not related to heart
* 2: Non-anginal pain: typically esophageal spasms (non heart related)
* 3: Asymptomatic: chest pain not showing signs of disease
4. trestbps - resting blood pressure (in mm Hg on admission to the hospital) anything above 130-140 is typically cause for concern
5. chol - serum cholestoral in mg/dl
* serum = LDL + HDL + .2 * triglycerides
* above 200 is cause for concern
6. fbs - (fasting blood sugar > 120 mg/dl) (1 = true; 0 = false)
* '>126' mg/dL signals diabetes
7. restecg - resting electrocardiographic results
* 0: Nothing to note
* 1: ST-T Wave abnormality
    * can range from mild symptoms to severe problems   
    * signals non-normal heart beat
* just 2: Possible or definite left ventricular hypertrophy
    * Enlarged heart's main pumping chamber
8. thalach - maximum heart rate achieved
9. exang - exercise induced angina (1 = yes; 0 = no)
10. oldpeak - ST depression induced by exercise relative to rest looks at stress of heart during excercise unhealthy heart will stress more
11. slope - the slope of the peak exercise ST segment
* 0: Upsloping: better heart rate with excercise (uncommon)
* 1: Flatsloping: minimal change (typical healthy heart)
* 2: Downslopins: signs of unhealthy heart
12. ca - number of major vessels (0-3) colored by flourosopy
* colored vessel means the doctor can see the blood passing through
* the more blood movement the better (no clots)
13. thal - thalium stress result
* 1,3: normal
* 6: fixed defect: used to be defect but ok now
* 7: reversable defect: no proper blood movement when excercising
14. target - have disease or not (1=yes, 0=no) (= the predicted attribute

# Preparing the tools
We're going to use pandas, matplotlib, numpy  for data analysis and manipulation

In [ ]:
# Import all the tools we need

# Regular EDA (exploratory data analysis) and plotting libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# we want our plots to appear inside the notebook
%matplotlib inline 

# Models from Scikit-Learn
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Model Evaluations
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import RocCurveDisplay

## Load data

In [ ]:
df = pd.read_csv("heart-disease.csv")
df.shape # Rows: 303 Columns: 14

## Data exploration (exploratory data analysis or EDA)
The goal here is to find out more about the data and become a subejct matter expert on the dataset you're working with.

1. What question(s) are we trying to solve?
2. What kind of data do we have and how do we treat differentn types?
3. What's missing from the data and how do you deal with it?
4. Where are the outliers and why should we care about them?
5. How can you add, change or remove features to get more of your data?

In [ ]:
df.head(), df.tail()

In [ ]:
# let's find out how many of each class there is
df["target"].value_counts()

In [ ]:
df["target"].value_counts().plot(kind="bar", color=["salmon", "lightblue"])

In [ ]:
df.info()

In [ ]:
# Are there any missing values?
df.isna().sum()

In [ ]:
df.describe()

### Heart Disease Frequency according to Sex

In [ ]:
df.sex.value_counts() # 1 is male and 0 is female

In [ ]:
# Compare target col with sex col
pd.crosstab(df["sex"], df["target"])

In [ ]:
# Create a plot of crosstab
from numpy import rot90


pd.crosstab(df["target"], df["sex"]).plot(
    kind="bar", 
    color=["salmon", "lightblue"])

plt.title("Heart Disease vs. Sex")
plt.xlabel("0 = No disease, 1 = Disease")
plt.ylabel("Amount")
plt.legend(["Female", "Male"])
# plt.legend([0, 1], ["Female", "Male"])
plt.xticks(rotation = 0)

### Age vs. Max Heart Rate for Heart Disease

In [ ]:
# Create another figure
plt.figure(figsize=(10, 6))

# Scatter with positive examples
plt.scatter(df.age[df.target==1],
            df.thalach[df.target==1],
            color='salmon',
            label='Heart Disease')
# Scatter with negative examples
plt.scatter(df.age[df.target==0],
            df.thalach[df.target==0],
            color='lightblue',
            label='Healthy')

# Add some helpful info
plt.title("Heart Disease in function of age and max heart rate")
plt.xlabel("Age")
plt.ylabel("Max heart rate")
plt.legend(["Disease", "Healthy"])

In [ ]:
# check the distribution of the age column with a histogram
df.age.plot.hist(bins=20, color="lightblue")

### Heart disease frequency per chest pain type:
3. cp - chest pain type:
    * 0: Typical angine: chest pain related decrease blood supply to the heart
    * 1: Atypical angina: chest pain not related to heart
    * 2: Non-anginal pain: typically esophageal spasms (non heart related)
    * 3: Asymptomatic: chest pain not showing signs of disease

In [ ]:
pd.crosstab(df.cp, df.target)

In [ ]:
# Make the crosstab more visual
pd.crosstab(df.cp, df.target).plot(kind="bar", color=["salmon", "lightblue"])

## Add more details
plt.title("Heart Disease frequency per chest pain type")
plt.xlabel("Chest pain type")
plt.ylabel("Frequency")
plt.legend(["Healthy", "Disease"])
plt.xticks(rotation = 0)

In [ ]:
# Make a correlation matrix
df.corr()

In [ ]:
# let's make the correlation matrix a little prettier
corr_matrix = df.corr()
fig, ax = plt.subplots(figsize=(15, 10))
ax = sns.heatmap(corr_matrix, annot=True, linewidths=0.5, fmt=".2f", cmap="YlGnBu")

## 5. Modeling

In [ ]:
df.head()

In [ ]:
# split data into X and y
X = df.drop("target", axis=1)
y = df["target"]
X

In [ ]:
y

In [ ]:
# Split data into train and test sets
np.random.seed(42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train

Now we're got out data split into trainign and test sets, it's time to build a machine learning model.

We'll train it (find the patterns) on the training set.

And we'll test it (use the patterns) on the test set.

We're going to try 3 different machine learning models:
1. Logistic Regression
2. K-Nearest Neighbours Classifier
3. Random Forest Classifier

In [ ]:
# Put mdoels in a dictionary
models = {"Logistic Regression": LogisticRegression(),
          "KNN": KNeighborsClassifier(),
          "Random Forest": RandomForestClassifier()}

# Create a function to fit and score models
def fit_and_score(models, X_train, X_test, y_train, y_test):
    np.random.seed(42)
    model_scores = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        print(f"{name} accuracy is: {model.score(X_test, y_test)}")
        model_scores[name] = model.score(X_test, y_test)
    return model_scores


In [ ]:
model_scores = fit_and_score(models, X_train, X_test, y_train, y_test)
model_scores

### Model Comparison

In [ ]:
model_compare = pd.DataFrame(model_scores, index=["Accuracy"])
model_compare.T

In [ ]:
model_compare.T.plot(kind="bar", rot=0);

Now we're got out base model, and we know a model's first predictions aren't always what we should base our next steps off.

Let's look at the following:
* Hyperparameter tunig
* Feature importance
* Confusion matrix
* Cross-validaiton
* Precision
* Recall
* F1 score
* Classification report
* ROC curve
* Area under the curse (AUC)


### Hyperparameter tuning

In [ ]:
# Let's tune KNN

from sklearn import neighbors


train_scores = []
test_scores = []

# Create a list of different values for n_neighbours
neighbors = list(range(1, 21))

# Setup KNN instance
knn = KNeighborsClassifier()

# Loop over different values of k
for k in neighbors:
    # Setup a k-NN classifier with k neighbors: knn
    knn = KNeighborsClassifier(n_neighbors=k)
    # Fit the classifier to the training data
    knn.fit(X_train, y_train)
    # Compute score on the training set
    train_scores.append(knn.score(X_train, y_train))
    # Compute score on the testing set
    test_scores.append(knn.score(X_test, y_test))

In [ ]:
train_scores

In [ ]:
test_scores

In [ ]:
from scipy.stats import f


plt.plot(neighbors, train_scores, label="Train scores")
plt.plot(neighbors, test_scores, label="Test scores")
plt.xticks(neighbors)
plt.xlabel("number of neighbors")
plt.ylabel("model score")
plt.legend()

print(f"Maximum KNN score one the test data is {np.max(test_scores)*100:.2f}%")
print(f"Optimal number of neighbors is {neighbors[np.argmax(test_scores)]}")

## Hyperparameter tuning with RandomizedSearchCV
We're going to tune:
* LogisticRegression()
* RandomForestClassifier()

using RandomizedSearchCV

In [ ]:
# Create a hyperparameter grid for LogisticRegression
log_reg_grid = {
    "C": np.logspace(-4, 4, 20),
    "solver": ["liblinear"],
    }

# Createa hyperparameter grid for random forest
rf_grid = {
    "n_estimators": np.arange(10, 1000, 50),
    "max_features": [1, 2, 3],
    "max_depth": [None, 3, 5, 10],
    "min_samples_split": np.arange(2, 20, 2),
    "min_samples_leaf": np.arange(1, 20, 2), 
    }

Now we've got hyperparameter grids setup for each of our models, let's tune them using RandomizedSearchCV

In [ ]:
np.random.seed(42)

# setup random hyperparameter search for logisticRegression
rs_log_reg = RandomizedSearchCV(
    estimator=LogisticRegression(),
    param_distributions=log_reg_grid,
    n_iter=20,
    cv=5,
    verbose=1,
)

# Fit random hyperparameter search model for LogisticRegression
rs_log_reg.fit(X_train, y_train)


In [ ]:
rs_log_reg.best_params_

In [ ]:
rs_log_reg.score(X_test, y_test)

Now we've tuned LogisticRegression(), let's do the same for RandomForestClassifier()

In [ ]:
np.random.seed(42)

# Setup random hyperparameter search for RandomForestClassifier
rs_rf = RandomizedSearchCV(
    estimator=RandomForestClassifier(),
    param_distributions=rf_grid,
    n_iter=20,
    cv=5,
    verbose=1,
)

# Fit random hyperparameter search model for RandomForestClassifier
rs_rf.fit(X_train, y_train);

In [ ]:
# Find the best hyperparameters
rs_rf.best_params_

In [ ]:
# Evaluate the model
rs_rf.score(X_test, y_test)

In [ ]:
model_scores

## Hyperparameter Tuning with GridSearchCV
Since our LogisticRegression model provides the best scores so far, we'll try the GridSearchCV

In [ ]:
# Different hyperparameters for our LogisticRegression model
log_reg_grid = {
    "C": np.logspace(-4, 4, 30),
    "solver": ["liblinear"],
    }

# Setup grid hyperparameter search for LogisticRegression
gs_log_reg = GridSearchCV(
    estimator=LogisticRegression(),
    param_grid=log_reg_grid,
    cv=5,
    verbose=1,
    n_jobs=-1
)

# Fit grid hyperparameter search model
gs_log_reg.fit(X_train, y_train);

In [ ]:
gs_log_reg.best_params_

In [ ]:
# Evaluate the grid search LogisticRegression model
gs_log_reg.score(X_test, y_test)

## Evaluating our tuned machine learning classifier, beyond accuracy
* ROC curse and AUC score
* Confusion matrix
* Classification report
* Precision
* Recall
* F1-score

and it would be great if cross-validation was used where possible.

To make comparisons and evaluate our trained model, first we need to make predictions

In [ ]:
# Make predictions with tuned model
y_preds = gs_log_reg.predict(X_test)
y_preds

In [ ]:
y_test

In [ ]:
# Plot ROC curve and calcualte AUC metric
RocCurveDisplay.from_estimator(gs_log_reg, X_test, y_test)

In [ ]:
# Confusion matrix
print(confusion_matrix(y_test, y_preds))

In [ ]:

def plot_conf_mat(y_test, y_preds):
    conf_mat = confusion_matrix(y_test, y_preds)
    sns.heatmap(conf_mat, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.show()
    
plot_conf_mat(y_test, y_preds)

Now we've got a ROC curve, an AUC metric and a confusion matrix. Let's get a classification report as well as cross-validated precision, recall and F1-score.

In [ ]:
print(classification_report(y_test, y_preds))

### Calculate evaluation metrics using cross-validation
We're going to calculate precision, recall, and F1-score of our model using cross-validation and to do so we'll be using `cross_val_score()`

In [ ]:
# Check best hyperparameters
gs_log_reg.best_params_

In [ ]:
# Create a new classifier with best params
clf = LogisticRegression(C=0.20433597178569418, solver='liblinear') 

In [ ]:
# Cross validated accuracy
cv_acc = cross_val_score(clf, X_train, y_train, cv=5, scoring = "accuracy")
print(f"Cross validated accuracy is: {np.mean(cv_acc)}")

In [ ]:
# Cross validated precision
cv_prec = cross_val_score(clf, X_train, y_train, cv=5, scoring = "precision")
print(f"Cross validated precision is: {np.mean(cv_prec)}")

In [ ]:
# Cross validated recall
cv_rec = cross_val_score(clf, X_train, y_train, cv=5, scoring = "recall")
print(f"Cross validated recall is: {np.mean(cv_rec)}")

In [ ]:
# Cross validated F1 score
cv_f1 = cross_val_score(clf, X_train, y_train, cv=5, scoring = "f1")
print(f"Cross validated F1 score is: {np.mean(cv_f1)}")

In [ ]:
# visualize cross-validated metrics
cv_metrics = pd.DataFrame(
    {
        "Accuracy": np.mean(cv_acc),
        "Precision": np.mean(cv_prec),
        "Recall": np.mean(cv_rec),
        "F1": np.mean(cv_f1),
    },
    index=[0]
)

In [ ]:

cv_metrics.T.plot.bar(title="Cross Validated Metrics", figsize=(10, 5), legend=False, rot=0)
plt.show()

### Feature importance
Feature importance is basically asking: "Which features contributed most to the outcomes of the model and how did they contribute?"

Finding feature importance is different for each machine learning model.

Let's find the feature importance for our logistic regression model.

In [ ]:
# Fit an instance of LogisticRegression to the data
clf = LogisticRegression(C=0.20433597178569418, solver='liblinear') 
clf.fit(X_train, y_train);

In [ ]:
# Check coef_
clf.coef_

In [ ]:
# Match coefs of features to columns
feature_dict = dict(zip(X_train.columns, clf.coef_[0]))
feature_dict

In [ ]:
# Visualize feature imoprtance
feature_df = pd.DataFrame(feature_dict, index=[0])
feature_df.T.plot.bar(title="Feature Importance", legend=False, rot=0)

## 6. Experimentation
If you haven't hit your evaluation metric yet, ask yourself:
* Could you collect more data?
* Could yoy try a better model? Like CatBoost or XGBoost?
* Could you improve the current mdoels? (beyond what we're done so far)
* If your model is good enough (you have hit your evaluation metric) how would you export it and share it with others?
